In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/KaggleV2-May-2016.csv")

# target
df['No_show'] = df['No-show'].map({'No': 0, 'Yes': 1})

# dates → features
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['waiting_days'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days.clip(lower=0)
df['appointment_weekday'] = df['AppointmentDay'].dt.dayofweek

# choose simple numeric features (enough for Phase 2)
features = [
    'Age','Scholarship','Hipertension','Diabetes','Alcoholism','Handcap',
    'SMS_received','waiting_days','appointment_weekday'
]
X = df[features].astype(float).values
y = df['No_show'].values.astype(float)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


In [5]:
import torch, torch.nn as nn, torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32).to(device)

class NoShowNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = NoShowNet(X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# sanity: overfit a single batch
batch_X = X_train_t[:128]
batch_y = y_train_t[:128]

for epoch in range(50):
    optimizer.zero_grad()
    out = model(batch_X)
    loss = criterion(out, batch_y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"sanity epoch {epoch:02d} | loss {loss.item():.4f}")


sanity epoch 00 | loss 0.6847
sanity epoch 10 | loss 0.6114
sanity epoch 20 | loss 0.5483
sanity epoch 30 | loss 0.4934
sanity epoch 40 | loss 0.4486


In [6]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir="logs/run1")

# a tiny real training loop over the whole train set (few epochs is enough for Phase 2)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val.reshape(-1,1), dtype=torch.float32).to(device)

for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    out = model(X_train_t)
    loss = criterion(out, y_train_t)
    loss.backward()
    optimizer.step()

    # validation
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, y_val_t)

    writer.add_scalar("loss/train", loss.item(), epoch)
    writer.add_scalar("loss/val",   val_loss.item(), epoch)
    print(f"epoch {epoch:02d} | train {loss.item():.4f} | val {val_loss.item():.4f}")

writer.close()

# checkpoint + config
import json, os
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/no_show_checkpoint.pt")
with open("logs/config.json","w") as f:
    json.dump({"lr":1e-3,"batch_size":"full-batch","epochs":10}, f)


epoch 00 | train 0.5134 | val 0.5135
epoch 01 | train 0.5133 | val 0.5133
epoch 02 | train 0.5134 | val 0.5130
epoch 03 | train 0.5129 | val 0.5126
epoch 04 | train 0.5127 | val 0.5121
epoch 05 | train 0.5120 | val 0.5114
epoch 06 | train 0.5119 | val 0.5105
epoch 07 | train 0.5107 | val 0.5096
epoch 08 | train 0.5102 | val 0.5085
epoch 09 | train 0.5091 | val 0.5074
